![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [2]:
from QuantConnect.Research import QuantBook
from QuantConnect.DataSource import CryptoUniverse
from QuantConnect import Resolution
from datetime import timedelta
from QuantConnect.Indicators import RateOfChange
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from QuantConnect.Research import QuantBook
from QuantConnect.DataSource import CryptoUniverse
from QuantConnect import Resolution
from statsmodels.tsa.stattools import coint
import pandas as pd
import numpy as np
from itertools import combinations
from datetime import datetime
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
from AlgorithmImports import *
import numpy as np

qb = QuantBook()

symbols_list = ["BTCUSDT"]

start = pd.Timestamp(2023, 1, 1, tz="UTC") 
end = pd.Timestamp(2025, 12, 31, tz="UTC")  # 3 months only

print("="*80)
print(f"Date Range: {start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}")
print("="*80)

symbols = []
for ticker in symbols_list:
    symbol = qb.add_crypto(ticker, market=Market.BINANCE).symbol
    symbols.append(symbol)

# ===== STEP 1: GET MINUTE DATA =====
minute_data_raw = qb.history(TradeBar, symbols, start, end, Resolution.Minute)

print("\n" + "="*80)
print("WORKING WITH 1-MINUTE BARS DIRECTLY (NO RESAMPLING)")
print("="*80)

# Reset index to get clean dataframe
minute_data_reset = minute_data_raw.reset_index()

# Process by symbol
dfs_by_symbol = {}
for sym in symbols:
    sym_data = minute_data_reset[minute_data_reset['symbol'] == sym].set_index('time')
    sym_data = sym_data[['open', 'high', 'low', 'close', 'volume']]
    dfs_by_symbol[sym] = sym_data

print("\nSymbols in dfs_by_symbol:", [sym.Value for sym in dfs_by_symbol.keys()])

# ===== STEP 2: CALCULATE TOTAL RETURN FROM 1-MIN BARS =====
btc_symbol = symbols[0]
btc_1min_df = dfs_by_symbol[btc_symbol].copy()

print(f"\n1-minute bars: {len(btc_1min_df):,}")

# Hold for 15 minutes = 15 one-minute bars
n_bars = 15  

def calculate_total_return_for_1min_bars(one_min_df, n_bars=15):
    """
    For each 1-min bar at position i, calculate total return over NEXT n_bars (future window)
    
    Example: At 1-min bar 04:10:00 (with n_bars=15):
      - Covers FUTURE 1-min bars: 04:11, 04:12, ..., 04:25
      - Return = (close at 04:25 - open at 04:11) / open at 04:11
      - This return is stored at the 04:10 timestamp
    """
    metrics_list = []
    
    for i in range(len(one_min_df) - n_bars):
        current_bar = one_min_df.iloc[i]
        start_bar = one_min_df.iloc[i + 1]  # Start from NEXT bar
        end_bar = one_min_df.iloc[i + n_bars]  # End n_bars ahead
        
        # Total return: open of first future 1-min bar to close of last future 1-min bar
        total_ret = (end_bar['close'] - start_bar['open']) / start_bar['open']
        
        metrics_list.append({
            'time': one_min_df.index[i],
            'total_return': total_ret
        })
    
    return pd.DataFrame(metrics_list).set_index('time')

# Calculate total return
print(f"\nCalculating total return over next {n_bars} one-min bars (15 minutes)...")
btc_metrics = calculate_total_return_for_1min_bars(btc_1min_df, n_bars=n_bars)
print(f"Total returns calculated: {len(btc_metrics):,}")

# Merge metrics onto 1-min dataframe
btc_1min_df = btc_1min_df.join(btc_metrics, how='left')

# Update dfs_by_symbol
dfs_by_symbol[btc_symbol] = btc_1min_df

print(f"\n1-minute bars with total return: {len(btc_1min_df):,}")

# ===== PRINT FIRST 20 BARS FOR VERIFICATION =====
print("\n" + "="*80)
print("FIRST 20 BARS: OHLC + TOTAL_RETURN (FUTURE 15-MIN RETURNS)")
print("="*80)

display_cols = ['open', 'high', 'low', 'close', 'total_return']
print(btc_1min_df[display_cols].head(20).to_string())

print("\n" + "="*80)
print(f"Dataset ready with {len(btc_1min_df):,} 1-minute bars")
print("="*80)

In [4]:
# ===== STEP 3: COMPUTE ALL FEATURES FOR BTC ON 1-MIN BARS =====
print("\n" + "="*80)
print("COMPUTING FEATURES ON 1-MINUTE BARS")
print("="*80)

def compute_btc_features_1min(df, n_bars=15):
    """
    Compute all selected features for BTC trading strategy on 1-minute bars.
    n_bars: number of 1-min bars for holding period (default 15 = 15 minutes)
    
    Window conversions from 5-min to 1-min bars:
    - 3 five-min bars = 15 one-min bars (15 minutes)
    - 24 five-min bars = 120 one-min bars (2 hours)
    - 48 five-min bars = 240 one-min bars (4 hours)
    - 288 five-min bars = 1440 one-min bars (24 hours)
    """
    df = df.copy()
    
    # ============================================================
    # BOLLINGER BAND WIDTH FEATURES
    # ============================================================
    # 2-hour BB width (120 one-min bars = 2 hours)
    sma_120 = df['close'].rolling(120).mean()
    std_120 = df['close'].rolling(120).std()
    df['bb_width_2h'] = (2 * std_120) / (sma_120 + 1e-8)
    
    # 4-hour BB width (240 one-min bars = 4 hours)
    sma_240 = df['close'].rolling(240).mean()
    std_240 = df['close'].rolling(240).std()
    df['bb_width_4h'] = (2 * std_240) / (sma_240 + 1e-8)
    
    # ============================================================
    # INTRABAR POSITION FEATURES
    # ============================================================
    # Close position within bar (0 = at low, 1 = at high)
    df['close_position'] = np.where(
        df['high'] != df['low'],
        (df['close'] - df['low']) / (df['high'] - df['low']),
        0.5
    )
    
    # High-low range (normalized by close)
    df['hl_range'] = (df['high'] - df['low']) / df['close']
    
    # ============================================================
    # DISTANCE TO HIGH/LOW FEATURES
    # ============================================================
    # 2-hour window (120 one-min bars)
    high_2h = df['close'].rolling(120).max()
    df['dist_2h_high'] = (df['close'] - high_2h) / high_2h
    
    # 1-day window (1440 one-min bars = 24 hours)
    high_1d = df['close'].rolling(1440).max()
    low_1d = df['close'].rolling(1440).min()
    df['dist_1d_high'] = (df['close'] - high_1d) / high_1d
    df['dist_1d_low'] = (df['close'] - low_1d) / low_1d
    
    # ============================================================
    # PATTERN FEATURES
    # ============================================================
    # Higher high pattern (15 one-min bars = 15 minutes)
    df['higher_high_3'] = (df['high'] > df['high'].shift(15)).astype(int)
    
    # ============================================================
    # VOLATILITY FEATURES
    # ============================================================
    # Calculate returns first
    df['ret_1'] = df['close'].pct_change(1)
    
    # 15-bar realized volatility (15 minutes)
    df['rvol_3'] = df['ret_1'].rolling(15).std()
    
    # 120-bar realized volatility (2 hours)
    rvol_120 = df['ret_1'].rolling(120).std()
    
    # 1440-bar realized volatility (1 day)
    rvol_1440 = df['ret_1'].rolling(1440).std()
    
    # Volatility ratio (2h vs 1d)
    df['rvol_ratio_24_288'] = rvol_120 / (rvol_1440 + 1e-8)
    
    # ============================================================
    # DROP OHLCV AND TEMPORARY COLUMNS
    # ============================================================
    df = df.drop(['open', 'high', 'low', 'close', 'volume', 'ret_1'], axis=1, errors='ignore')
    
    return df

# Calculate features for BTC
btc_symbol = symbols[0]
print(f"\nCalculating features for {btc_symbol.Value} on 1-minute bars...")
btc_df = dfs_by_symbol[btc_symbol].copy()

# Compute features
btc_df_features = compute_btc_features_1min(btc_df, n_bars=15)

# Update the dictionary
dfs_by_symbol[btc_symbol] = btc_df_features

# ===== VERIFICATION =====
print("\n" + "="*80)
print("FEATURE SUMMARY")
print("="*80)

final_df = dfs_by_symbol[btc_symbol]
print(f"Total rows: {len(final_df):,}")
print(f"Date range: {final_df.index.min()} to {final_df.index.max()}")

# List all features
feature_cols = [
    'total_return', 'bb_width_2h', 'bb_width_4h', 'close_position',
    'dist_1d_high', 'dist_1d_low', 'dist_2h_high', 'higher_high_3',
    'hl_range', 'rvol_3', 'rvol_ratio_24_288'
]

print("\nFeature Statistics:")
print("-" * 60)
for col in feature_cols:
    if col in final_df.columns:
        non_nan = final_df[col].notna().sum()
        nan_count = final_df[col].isna().sum()
        mean_val = final_df[col].mean()
        print(f"{col:25s} | Non-NaN: {non_nan:>8,} | NaN: {nan_count:>8,} | Mean: {mean_val:>10.6f}")

print("\n" + "="*80)
print("FEATURES COMPUTED ON 1-MINUTE BARS (NO RESAMPLING)")
print("="*80)

In [5]:
# ===== DATA CHECK: COMPARE FEATURES AND OHLC ON 1-MIN BARS =====
print("\n" + "="*80)
print("DATA CHECK - Starting 2024-01-02 08:00 UTC (1-MINUTE BARS)")
print("="*80)

# Get the data
btc_symbol = symbols[0]
btc_features = dfs_by_symbol[btc_symbol].copy()
btc_ohlc = btc_1min_df[['open', 'high', 'low', 'close']].copy()

# Merge everything
btc_full = btc_features.join(btc_ohlc, how='inner')

# Calculate forward return (15 one-min bars ahead = 15 minutes)
btc_full['forward_return'] = (btc_full['close'].shift(-15) - btc_full['close']) / btc_full['close']

# Filter using string slicing (avoids timezone issues)
data_check = btc_full['2024-01-02 08:00:00':'2024-01-02 12:00:00'].copy()

print(f"\nShowing data from 2024-01-02 08:00 to 12:00 UTC")
print(f"Total bars: {len(data_check)} (1-minute resolution)")

# Select columns to display
display_cols = ['open', 'high', 'low', 'close', 'rvol_3', 'rvol_ratio_24_288', 'forward_return']

print("\n" + "="*80)
print("OHLC + FEATURES + FORWARD RETURN (15 one-min bars = 15 minutes)")
print("="*80)

# Show first 5 bars
print("\nShowing first 5 bars:")
print(data_check[display_cols].head(5).to_string())

# Highlight trades based on rule
print("\n" + "="*80)
print("SIGNALS (rvol_3 > 0.001 AND rvol_ratio > 1.087)")
print("="*80)

signals = data_check[
    (data_check['rvol_3'] > 0.001) & 
    (data_check['rvol_ratio_24_288'] > 1.087)
].copy()

if len(signals) > 0:
    print(f"\nFound {len(signals)} signals in this period:")
    print(f"Showing first 20 signals:")
    print(signals[display_cols].head(20).to_string())
else:
    print("\nNo signals found in this period")

print("\n" + "="*80)
print("DATA CHECK COMPLETE")
print("="*80)

In [6]:
# ===== SIMPLE RULE BACKTEST IN RESEARCH (1-MINUTE BARS) =====
print("\n" + "="*80)
print("SIMPLE RULE BACKTEST - 1-MINUTE BARS")
print("="*80)

# Get features and OHLC
btc_symbol = symbols[0]
btc_features = dfs_by_symbol[btc_symbol].copy()
btc_ohlc = btc_1min_df[['open', 'high', 'low', 'close']].copy()

# Merge everything
btc_full = btc_features.join(btc_ohlc, how='inner')

print(f"\nData range: {btc_full.index.min()} to {btc_full.index.max()}")
print(f"Total bars: {len(btc_full):,}")

# ===== BACKTEST PARAMETERS =====
initial_capital = 100000  # $100k
hold_period = 15  # Hold for 15 one-minute bars (15 minutes)

# Simple rule thresholds
rvol3_threshold = 0.001
rvol_ratio_threshold = 1.087

# ===== RUN BACKTEST =====
trades = []
position = None  # Current position: None or dict with entry info
bars_held = 0

for i in range(len(btc_full)):
    row = btc_full.iloc[i]
    timestamp = btc_full.index[i]
    
    # Skip if features are NaN
    if pd.isna(row['rvol_3']) or pd.isna(row['rvol_ratio_24_288']):
        continue
    
    # If in position, check if we should exit
    if position is not None:
        bars_held += 1
        
        # Exit after holding period
        if bars_held >= hold_period:
            exit_price = row['close']
            exit_time = timestamp
            
            # Calculate return
            trade_return = (exit_price - position['entry_price']) / position['entry_price']
            
            trades.append({
                'entry_time': position['entry_time'],
                'exit_time': exit_time,
                'entry_price': position['entry_price'],
                'exit_price': exit_price,
                'return': trade_return,
                'bars_held': bars_held
            })
            
            # Reset position
            position = None
            bars_held = 0
        
        continue
    
    # Check for entry signal (only if not in position)
    if row['rvol_3'] > rvol3_threshold and row['rvol_ratio_24_288'] > rvol_ratio_threshold:
        # Enter long position at close of current bar
        position = {
            'entry_time': timestamp,
            'entry_price': row['close']
        }
        bars_held = 0

# ===== CALCULATE STATISTICS =====
if len(trades) > 0:
    trades_df = pd.DataFrame(trades)
    
    winning_trades = trades_df[trades_df['return'] > 0]
    losing_trades = trades_df[trades_df['return'] <= 0]
    
    total_trades = len(trades_df)
    win_rate = len(winning_trades) / total_trades * 100
    avg_return = trades_df['return'].mean() * 100
    total_return = trades_df['return'].sum() * 100
    
    # Sharpe ratio (annualized, assuming ~525,600 minutes per year)
    returns_std = trades_df['return'].std()
    if returns_std > 0:
        sharpe = (avg_return / 100) / returns_std * np.sqrt(525600 / hold_period)
    else:
        sharpe = 0
    
    # Calculate trades per day
    date_range = (btc_full.index.max() - btc_full.index.min()).days
    trades_per_day = total_trades / date_range if date_range > 0 else 0
    
    print("\n" + "="*80)
    print("BACKTEST RESULTS")
    print("="*80)
    print(f"\nTotal trades: {total_trades:,}")
    print(f"Winning trades: {len(winning_trades):,}")
    print(f"Losing trades: {len(losing_trades):,}")
    print(f"Win rate: {win_rate:.2f}%")
    print(f"\nAvg return per trade: {avg_return:.4f}%")
    print(f"Total return: {total_return:.2f}%")
    print(f"Best trade: {trades_df['return'].max()*100:.4f}%")
    print(f"Worst trade: {trades_df['return'].min()*100:.4f}%")
    print(f"\nSharpe ratio: {sharpe:.2f}")
    print(f"\nBacktest period: {date_range} days")
    print(f"Trades per day: {trades_per_day:.2f}")
    
    # Show first 10 trades
    print("\n" + "="*80)
    print("FIRST 10 TRADES")
    print("="*80)
    trades_df['return_pct'] = trades_df['return'] * 100
    print(trades_df[['entry_time', 'exit_time', 'entry_price', 'exit_price', 'return_pct', 'bars_held']].head(10).to_string(index=False))
    
else:
    print("\nNo trades generated!")

print("\n" + "="*80)

In [27]:
# ===== DECISION TREE WITH CLASS BALANCING =====
print("\n" + "="*80)
print("DECISION TREE PATTERN DISCOVERY - BALANCED VERSION")
print("="*80)

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# ===== STEP 1: PREPARE DATA WITH BETTER TARGET =====
print("\nStep 1: Preparing Data with Better Target Definition")
print("-" * 80)

btc_symbol = symbols[0]
btc_features = dfs_by_symbol[btc_symbol].copy()

feature_cols = [
    'bb_width_2h', 'bb_width_4h', 'close_position', 'hl_range',
    'dist_2h_high', 'dist_1d_high', 'dist_1d_low', 
    'higher_high_3', 'rvol_3', 'rvol_ratio_24_288'
]

# NEW TARGET: Top 20% of returns (more balanced)
return_percentile = 80  # Top 20%
threshold = btc_features['total_return'].quantile(return_percentile / 100)

btc_features['target'] = (btc_features['total_return'] > threshold).astype(int)

data = btc_features[feature_cols + ['target', 'total_return']].dropna()

print(f"Total samples: {len(data):,}")
print(f"Target threshold: {threshold*100:.4f}% (top {100-return_percentile}%)")
print(f"Target distribution:")
print(f"  Positive (top 20%): {(data['target'] == 1).sum():,} ({(data['target'] == 1).mean()*100:.2f}%)")
print(f"  Negative (bottom 80%): {(data['target'] == 0).sum():,} ({(data['target'] == 0).mean()*100:.2f}%)")

# ===== STEP 2: TRAIN/TEST SPLIT =====
print("\nStep 2: Train/Test Split")
print("-" * 80)

split_idx = int(len(data) * 0.8)

train_data = data.iloc[:split_idx]
test_data = data.iloc[split_idx:]

X_train = train_data[feature_cols].values
y_train = train_data['target'].values
returns_train = train_data['total_return'].values

X_test = test_data[feature_cols].values
y_test = test_data['target'].values
returns_test = test_data['total_return'].values

print(f"Train: {len(X_train):,}, Test: {len(X_test):,}")

# ===== STEP 3: TRAIN WITH CLASS WEIGHTS =====
print("\nStep 3: Training Decision Tree with Class Balancing")
print("-" * 80)

# Calculate class weights to balance
neg_weight = len(y_train) / ((y_train == 0).sum() * 2)
pos_weight = len(y_train) / ((y_train == 1).sum() * 2)

print(f"Class weights: Negative={neg_weight:.3f}, Positive={pos_weight:.3f}")

tree = DecisionTreeClassifier(
    max_depth=6,  # Allow deeper tree
    min_samples_leaf=50,  # Smaller leaf size
    min_impurity_decrease=0.0001,  # Smaller threshold
    class_weight={0: neg_weight, 1: pos_weight},  # BALANCE CLASSES
    random_state=42
)

tree.fit(X_train, y_train)

print(f"Tree depth: {tree.get_depth()}")
print(f"Number of leaves: {tree.get_n_leaves()}")

# ===== STEP 4: EVALUATE =====
print("\nStep 4: Evaluation")
print("-" * 80)

y_pred = tree.predict(X_test)
y_proba = tree.predict_proba(X_test)[:, 1]

from sklearn.metrics import accuracy_score, precision_score, recall_score
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred)

print(f"Accuracy: {acc*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall: {recall*100:.2f}%")
print(f"Max probability: {y_proba.max():.3f}")
print(f"Samples with >50% confidence: {(y_proba > 0.5).sum():,}")

# ===== STEP 5: HIGH-CONFIDENCE ANALYSIS =====
print("\n" + "="*80)
print("HIGH-CONFIDENCE TRADING STRATEGY")
print("="*80)

confidence_levels = [0.5, 0.6, 0.7, 0.8, 0.9]

print(f"\nPerformance at different confidence levels:")
print("-" * 100)
print(f"{'Confidence':<12} | {'Trades':<10} | {'Trade %':<10} | {'Win Rate':<10} | {'Avg Return':<12} | {'Total Return':<12} | {'Sharpe':<8}")
print("-" * 100)

best_conf = None
best_sharpe = 0

for conf_threshold in confidence_levels:
    high_conf_mask = y_proba > conf_threshold
    
    if high_conf_mask.sum() > 0:
        hc_returns = returns_test[high_conf_mask]
        n_trades = len(hc_returns)
        trade_pct = n_trades / len(returns_test) * 100
        win_rate = (hc_returns > 0).mean()
        avg_return = hc_returns.mean()
        total_return = hc_returns.sum()
        sharpe = (avg_return / hc_returns.std()) * np.sqrt(525600 / 15) if hc_returns.std() > 0 else 0
        
        print(f"{conf_threshold:<12.1f} | {n_trades:<10,} | {trade_pct:<10.2f} | {win_rate*100:<9.1f}% | {avg_return*100:<11.4f}% | {total_return*100:<11.2f}% | {sharpe:<8.2f}")
        
        if sharpe > best_sharpe:
            best_sharpe = sharpe
            best_conf = conf_threshold
    else:
        print(f"{conf_threshold:<12.1f} | {'0':<10} | {'0.00':<10} | {'-':<10} | {'-':<12} | {'-':<12} | {'-':<8}")

# ===== STEP 6: XGBOOST =====
print("\n" + "="*80)
print("XGBOOST (GRADIENT BOOSTED TREES)")
print("="*80)

print("Training XGBoost...")

# Calculate scale_pos_weight for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,  # Handle imbalance
    min_child_weight=50,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

print("Training complete!")

# Feature importance
xgb_importance = xgb_model.feature_importances_
importance_ranking = sorted(zip(feature_cols, xgb_importance), key=lambda x: x[1], reverse=True)

print(f"\nXGBoost Feature Importance:")
print("-" * 50)
for feat, imp in importance_ranking:
    print(f"{feat:<25} | {imp:<10.4f}")

# Predictions
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print(f"\n\nXGBoost Performance at different confidence levels:")
print("-" * 100)
print(f"{'Confidence':<12} | {'Trades':<10} | {'Trade %':<10} | {'Win Rate':<10} | {'Avg Return':<12} | {'Total Return':<12} | {'Sharpe':<8}")
print("-" * 100)

xgb_best_sharpe = 0
xgb_best_conf = None

for conf_threshold in confidence_levels:
    high_conf_mask = xgb_proba > conf_threshold
    
    if high_conf_mask.sum() > 0:
        hc_returns = returns_test[high_conf_mask]
        n_trades = len(hc_returns)
        trade_pct = n_trades / len(returns_test) * 100
        win_rate = (hc_returns > 0).mean()
        avg_return = hc_returns.mean()
        total_return = hc_returns.sum()
        sharpe = (avg_return / hc_returns.std()) * np.sqrt(525600 / 15) if hc_returns.std() > 0 else 0
        
        print(f"{conf_threshold:<12.1f} | {n_trades:<10,} | {trade_pct:<10.2f} | {win_rate*100:<9.1f}% | {avg_return*100:<11.4f}% | {total_return*100:<11.2f}% | {sharpe:<8.2f}")
        
        if sharpe > xgb_best_sharpe:
            xgb_best_sharpe = sharpe
            xgb_best_conf = conf_threshold
    else:
        print(f"{conf_threshold:<12.1f} | {'0':<10} | {'0.00':<10} | {'-':<10} | {'-':<12} | {'-':<12} | {'-':<8}")

# ===== SUMMARY =====
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print(f"""
RESULTS:

Decision Tree (balanced):
- Best confidence: {best_conf if best_conf else 'N/A'}
- Best Sharpe: {best_sharpe:.2f}

XGBoost:
- Best confidence: {xgb_best_conf if xgb_best_conf else 'N/A'}
- Best Sharpe: {xgb_best_sharpe:.2f}

TOP 3 MOST IMPORTANT FEATURES:
""")

for i, (feat, imp) in enumerate(importance_ranking[:3]):
    print(f"{i+1}. {feat}: {imp:.4f}")

print(f"""
RECOMMENDATION:
Use {'XGBoost' if xgb_best_sharpe > best_sharpe else 'Decision Tree'} with {xgb_best_conf if xgb_best_sharpe > best_sharpe else best_conf:.1f} confidence threshold

NEXT STEPS:
1. Backtest the best model
2. Extract specific rules from tree
3. Test on different time periods
4. Add risk management (stop loss, position sizing)
""")

print("="*80)

# Save results
results = {
    'tree': tree,
    'xgb': xgb_model,
    'feature_cols': feature_cols,
    'tree_proba': y_proba,
    'xgb_proba': xgb_proba,
    'test_data': test_data,
    'returns_test': returns_test,
    'feature_importance': dict(importance_ranking),
    'best_model': 'xgb' if xgb_best_sharpe > best_sharpe else 'tree'
}

In [30]:
# ===== ANALYZE 75% CONFIDENCE SIGNALS =====
print("\n" + "="*80)
print("ANALYZING XGBOOST 75% CONFIDENCE SIGNALS")
print("="*80)

# Get 75% confidence samples
conf_threshold = 0.75
high_conf_mask = results['xgb_proba'] > conf_threshold

print(f"Total 75%+ confidence signals: {high_conf_mask.sum()}")

if high_conf_mask.sum() > 0:
    # Get feature values for these signals
    test_features = test_data[feature_cols]
    high_conf_features = test_features[high_conf_mask]
    high_conf_returns = returns_test[high_conf_mask]
    high_conf_probas = results['xgb_proba'][high_conf_mask]
    
    print("\n" + "="*80)
    print("FEATURE STATISTICS AT 75% CONFIDENCE")
    print("="*80)
    
    # Compare to overall distribution
    print(f"\n{'Feature':<25} | {'Overall Mean':<15} | {'75% Conf Mean':<15} | {'Difference':<15}")
    print("-" * 75)
    
    for feat in feature_cols:
        overall_mean = test_features[feat].mean()
        conf_mean = high_conf_features[feat].mean()
        diff = ((conf_mean - overall_mean) / abs(overall_mean) * 100) if overall_mean != 0 else 0
        print(f"{feat:<25} | {overall_mean:<15.6f} | {conf_mean:<15.6f} | {diff:>+14.1f}%")
    
    print("\n" + "="*80)
    print("FEATURE RANGES AT 75% CONFIDENCE")
    print("="*80)
    
    print(f"\n{'Feature':<25} | {'Min':<15} | {'Median':<15} | {'Max':<15}")
    print("-" * 75)
    
    for feat in feature_cols:
        feat_min = high_conf_features[feat].min()
        feat_median = high_conf_features[feat].median()
        feat_max = high_conf_features[feat].max()
        print(f"{feat:<25} | {feat_min:<15.6f} | {feat_median:<15.6f} | {feat_max:<15.6f}")
    
    # Extract simple rules (approximate thresholds)
    print("\n" + "="*80)
    print("APPROXIMATE TRADING RULES (75% CONFIDENCE)")
    print("="*80)
    
    print("\nBased on feature ranges, the model looks for:")
    print("-" * 60)
    
    # Focus on top 3 most important features
    top_features = ['rvol_3', 'hl_range', 'bb_width_4h']
    
    for feat in top_features:
        feat_median = high_conf_features[feat].median()
        feat_25 = high_conf_features[feat].quantile(0.25)
        feat_75 = high_conf_features[feat].quantile(0.75)
        
        print(f"\n{feat}:")
        print(f"  Typical range: {feat_25:.6f} to {feat_75:.6f}")
        print(f"  Median: {feat_median:.6f}")
        
        # Compare to overall
        overall_median = test_features[feat].median()
        print(f"  Overall median: {overall_median:.6f}")
        
        if feat_median > overall_median * 1.2:
            print(f"  → Look for HIGH {feat} (above {feat_25:.6f})")
        elif feat_median < overall_median * 0.8:
            print(f"  → Look for LOW {feat} (below {feat_75:.6f})")
    
    # Performance breakdown
    print("\n" + "="*80)
    print("PERFORMANCE BREAKDOWN")
    print("="*80)
    
    winners = high_conf_returns > 0
    losers = high_conf_returns <= 0
    
    print(f"\nWinning trades ({winners.sum()}):")
    print(f"  Avg return: {high_conf_returns[winners].mean()*100:.4f}%")
    print(f"  Best: {high_conf_returns[winners].max()*100:.3f}%")
    
    print(f"\nLosing trades ({losers.sum()}):")
    print(f"  Avg return: {high_conf_returns[losers].mean()*100:.4f}%")
    print(f"  Worst: {high_conf_returns[losers].min()*100:.3f}%")
    
    print(f"\nReward/Risk ratio: {abs(high_conf_returns[winners].mean() / high_conf_returns[losers].mean()):.2f}")
    
    # Show some actual examples
    print("\n" + "="*80)
    print("SAMPLE 75% CONFIDENCE SIGNALS")
    print("="*80)
    
    # Get indices for high confidence samples
    high_conf_indices = np.where(high_conf_mask)[0]
    
    # Show 5 winning and 5 losing examples
    # First filter the returns, then get indices
    winners = high_conf_returns > 0
    losers = high_conf_returns <= 0
    
    winning_indices = high_conf_indices[winners][:5]
    losing_indices = high_conf_indices[losers][:5]
    
    print("\n5 WINNING SIGNALS:")
    print("-" * 100)
    for idx in winning_indices:
        timestamp = test_data.index[idx]
        proba = results['xgb_proba'][idx]
        ret = returns_test[idx]
        
        print(f"\nTime: {timestamp} | Probability: {proba:.3f} | Return: {ret*100:.3f}%")
        print(f"  rvol_3: {test_features.iloc[idx]['rvol_3']:.6f}")
        print(f"  hl_range: {test_features.iloc[idx]['hl_range']:.6f}")
        print(f"  bb_width_4h: {test_features.iloc[idx]['bb_width_4h']:.6f}")
    
    print("\n5 LOSING SIGNALS:")
    print("-" * 100)
    for idx in losing_indices:
        timestamp = test_data.index[idx]
        proba = results['xgb_proba'][idx]
        ret = returns_test[idx]
        
        print(f"\nTime: {timestamp} | Probability: {proba:.3f} | Return: {ret*100:.3f}%")
        print(f"  rvol_3: {test_features.iloc[idx]['rvol_3']:.6f}")
        print(f"  hl_range: {test_features.iloc[idx]['hl_range']:.6f}")
        print(f"  bb_width_4h: {test_features.iloc[idx]['bb_width_4h']:.6f}")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

print("""
SUMMARY:
The XGBoost model at 75% confidence has found a selective pattern
that predicts large positive returns with high accuracy.

Key insight: The model is primarily looking at rvol_3 (recent volatility)
combined with other features to identify explosive moves.

Next: Backtest this with realistic trading (no overlapping positions)
""")

In [32]:
# ===== EXTRACTED RULE FROM XGBOOST 75% CONFIDENCE =====
print("\n" + "="*80)
print("EXTRACTED TRADING RULE - BACKTEST")
print("="*80)

"""
Based on XGBoost 75% confidence analysis, the pattern is:

ENTRY CONDITIONS:
1. rvol_3 > 0.000959 (high recent volatility)
2. hl_range > 0.001293 (wide current bar range)
3. bb_width_4h > 0.009114 (expanded 4h Bollinger Bands)

These 3 conditions together identify explosive moves.
"""

# ===== STEP 1: LOAD DATA =====
print("\nStep 1: Loading Data")
print("-" * 80)

btc_symbol = symbols[0]
btc_features = dfs_by_symbol[btc_symbol].copy()
btc_ohlc = btc_1min_df[['open', 'high', 'low', 'close']].copy()

# Merge
btc_full = btc_features.join(btc_ohlc, how='inner')
btc_full['forward_return'] = (btc_full['close'].shift(-15) - btc_full['close']) / btc_full['close']
btc_full = btc_full.dropna()

print(f"Total bars: {len(btc_full):,}")
print(f"Date range: {btc_full.index.min()} to {btc_full.index.max()}")

# ===== STEP 2: DEFINE EXTRACTED RULES =====
print("\nStep 2: Defining Extracted Rules")
print("-" * 80)

# Thresholds from XGBoost 75% confidence analysis
rvol3_threshold = 0.000959
hl_range_threshold = 0.001293
bb_width_4h_threshold = 0.009114

print(f"Rule conditions:")
print(f"  1. rvol_3 > {rvol3_threshold:.6f}")
print(f"  2. hl_range > {hl_range_threshold:.6f}")
print(f"  3. bb_width_4h > {bb_width_4h_threshold:.6f}")

# ===== STEP 3: APPLY RULE =====
print("\nStep 3: Applying Rule")
print("-" * 80)

# Create signal
signal_mask = (
    (btc_full['rvol_3'] > rvol3_threshold) &
    (btc_full['hl_range'] > hl_range_threshold) &
    (btc_full['bb_width_4h'] > bb_width_4h_threshold)
)

print(f"Signals generated: {signal_mask.sum():,} / {len(signal_mask):,}")
print(f"Signal frequency: {signal_mask.sum() / len(signal_mask) * 100:.2f}%")

# ===== STEP 4: BACKTEST (NO OVERLAPPING POSITIONS) =====
print("\nStep 4: Backtesting with No Overlapping Positions")
print("-" * 80)

trades = []
in_position = False
bars_held = 0
hold_period = 15

for i in range(len(btc_full)):
    row = btc_full.iloc[i]
    timestamp = btc_full.index[i]
    
    # Skip if NaN
    if pd.isna(row['forward_return']):
        continue
    
    # If in position, count bars
    if in_position:
        bars_held += 1
        if bars_held >= hold_period:
            in_position = False
            bars_held = 0
        continue
    
    # Check for entry signal
    if signal_mask.iloc[i]:
        trades.append({
            'timestamp': timestamp,
            'entry_price': row['close'],
            'rvol_3': row['rvol_3'],
            'hl_range': row['hl_range'],
            'bb_width_4h': row['bb_width_4h'],
            'forward_return': row['forward_return']
        })
        in_position = True
        bars_held = 0

trades_df = pd.DataFrame(trades)

print(f"Total trades (no overlap): {len(trades_df):,}")

# ===== STEP 5: PERFORMANCE METRICS =====
print("\n" + "="*80)
print("PERFORMANCE METRICS")
print("="*80)

if len(trades_df) > 0:
    returns = trades_df['forward_return'].values
    
    winning_trades = (returns > 0).sum()
    losing_trades = (returns <= 0).sum()
    win_rate = winning_trades / len(returns)
    
    total_return = returns.sum()
    avg_return = returns.mean()
    median_return = np.median(returns)
    std_return = returns.std()
    
    best_trade = returns.max()
    worst_trade = returns.min()
    
    # Sharpe ratio
    sharpe = (avg_return / std_return) * np.sqrt(525600 / hold_period) if std_return > 0 else 0
    
    print(f"\nTrade Statistics:")
    print(f"  Total trades: {len(returns):,}")
    print(f"  Winning trades: {winning_trades:,} ({win_rate*100:.2f}%)")
    print(f"  Losing trades: {losing_trades:,} ({(1-win_rate)*100:.2f}%)")
    
    print(f"\nReturn Statistics:")
    print(f"  Total return: {total_return*100:.2f}%")
    print(f"  Average per trade: {avg_return*100:.4f}%")
    print(f"  Median per trade: {median_return*100:.4f}%")
    print(f"  Std dev: {std_return*100:.4f}%")
    print(f"  Best trade: {best_trade*100:.3f}%")
    print(f"  Worst trade: {worst_trade*100:.3f}%")
    
    print(f"\nRisk-Adjusted Performance:")
    print(f"  Sharpe Ratio: {sharpe:.2f}")
    
    # Reward/risk
    avg_win = returns[returns > 0].mean() if (returns > 0).sum() > 0 else 0
    avg_loss = returns[returns <= 0].mean() if (returns <= 0).sum() > 0 else 0
    reward_risk = abs(avg_win / avg_loss) if avg_loss != 0 else 0
    
    print(f"  Avg win: {avg_win*100:.4f}%")
    print(f"  Avg loss: {avg_loss*100:.4f}%")
    print(f"  Reward/Risk ratio: {reward_risk:.2f}")
    
    # Calculate trades per day
    date_range = (btc_full.index.max() - btc_full.index.min()).days
    trades_per_day = len(trades_df) / date_range if date_range > 0 else 0
    
    print(f"\nTrade Frequency:")
    print(f"  Backtest period: {date_range} days")
    print(f"  Trades per day: {trades_per_day:.2f}")
    
    # ===== STEP 6: SPLIT BY TIME PERIOD =====
    print("\n" + "="*80)
    print("PERFORMANCE BY TIME PERIOD")
    print("="*80)
    
    # Split into months
    trades_df['month'] = trades_df['timestamp'].dt.to_period('M')
    
    # Calculate monthly stats manually
    monthly_groups = trades_df.groupby('month')
    
    print(f"\nMonthly Breakdown:")
    print("-" * 60)
    print(f"{'Month':<15} | {'Trades':<10} | {'Avg Return':<12} | {'Win Rate':<10}")
    print("-" * 60)
    
    for month, group in monthly_groups:
        trades_count = len(group)
        avg_ret = group['forward_return'].mean()
        win_rate_month = (group['forward_return'] > 0).mean()
        print(f"{str(month):<15} | {trades_count:<10} | {avg_ret*100:<11.4f}% | {win_rate_month*100:<9.1f}%")
    
    # ===== STEP 7: FIRST 20 TRADES =====
    print("\n" + "="*80)
    print("FIRST 20 TRADES")
    print("="*80)
    
    print(trades_df[['timestamp', 'entry_price', 'rvol_3', 'hl_range', 'bb_width_4h', 'forward_return']].head(20).to_string(index=False))

else:
    print("\nNo trades generated!")

# ===== SUMMARY =====
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

if len(trades_df) > 0:
    print(f"""
EXTRACTED RULE PERFORMANCE:

CONDITIONS:
  rvol_3 > {rvol3_threshold:.6f} (recent volatility spike)
  AND hl_range > {hl_range_threshold:.6f} (wide current bar)
  AND bb_width_4h > {bb_width_4h_threshold:.6f} (expanded 4h bands)

RESULTS:
  Trades: {len(trades_df):,}
  Win Rate: {win_rate*100:.2f}%
  Avg Return: {avg_return*100:.4f}%
  Sharpe: {sharpe:.2f}
  Trades/Day: {trades_per_day:.2f}

INTERPRETATION:
This rule identifies moments when:
- Recent volatility is elevated (rvol_3)
- Current bar shows strong movement (hl_range)
- 4-hour volatility is expanding (bb_width_4h)

These conditions together signal explosive continuation moves.

NEXT STEPS:
1. Test on different time periods (out-of-sample)
2. Add stop loss (~0.3%) and take profit (~0.5%)
3. Implement in QuantConnect backtest
4. Paper trade before going live
""")

print("="*80)

In [14]:
# ===== EXTRACT LEARNED GATES FROM MODEL =====
"""
Goal: Extract the actual thresholds the model learned during training
The gating mechanism already found optimal cutoffs - let's see what they are!
"""

import numpy as np
import pandas as pd
import torch

print("\n" + "="*80)
print("LEARNED GATE ANALYSIS - WHAT THE MODEL DISCOVERED")
print("="*80)

# ===== STEP 1: INSPECT MODEL STRUCTURE =====
print("\nInspecting model structure...")
print("\nModel layers:")
for name, module in model.named_modules():
    if name:  # Skip the root module
        print(f"  {name}: {type(module).__name__}")

print("\n\nModel state dict keys:")
state_dict = model.state_dict()
for key, tensor in state_dict.items():
    print(f"  {key}: {tensor.shape}")

# Try to find gate layer - common names
gate_weight_key = None
gate_bias_key = None

for key in state_dict.keys():
    if 'gate' in key.lower() and 'weight' in key.lower():
        gate_weight_key = key
    if 'gate' in key.lower() and 'bias' in key.lower():
        gate_bias_key = key

if gate_weight_key and gate_bias_key:
    print(f"\n✓ Found gate layer:")
    print(f"  Weight key: {gate_weight_key}")
    print(f"  Bias key: {gate_bias_key}")
    gate_weights = state_dict[gate_weight_key].cpu().numpy()
    gate_bias = state_dict[gate_bias_key].cpu().numpy()
    print(f"  Gate weights shape: {gate_weights.shape}")
    print(f"  Gate bias shape: {gate_bias.shape}")
else:
    print("\n✗ Could not find gate layer in model")
    print("\nPlease check your model definition and update the script with correct layer names.")
    print("\nFor now, I'll analyze the gate activations you already computed...")
    gate_weights = None
    gate_bias = None

print(f"Number of features: {len(feature_cols)}")

# ===== STEP 2: ANALYZE GATE ACTIVATIONS =====
print(f"\n{'='*80}")
print("GATE ACTIVATION ANALYSIS")
print(f"{'='*80}\n")

"""
Since we have test_gates_np (gate activations), we can analyze:
1. Which gates activate most often
2. What feature values trigger each gate
3. Which combinations work best
"""

gate_info = []

high_conf_mask = test_proba_np > 0.70
hc_feature_values = test_data.loc[test_data.index[high_conf_mask], feature_cols]
hc_gates = test_gates_np[high_conf_mask]
hc_returns = returns_test[high_conf_mask]

for i, feature in enumerate(feature_cols):
    gate_activations = hc_gates[:, i]
    feature_vals = hc_feature_values[feature].values
    
    # When is this gate active?
    active_mask = gate_activations > 0.5
    activation_rate = active_mask.mean() * 100
    
    # Feature values when gate is active vs inactive
    if active_mask.sum() > 10:
        active_mean = feature_vals[active_mask].mean()
        active_median = np.median(feature_vals[active_mask])
        active_min = feature_vals[active_mask].min()
        active_max = feature_vals[active_mask].max()
        active_25th = np.percentile(feature_vals[active_mask], 25)
        active_75th = np.percentile(feature_vals[active_mask], 75)
        
        # Compare to when gate is NOT active
        inactive_mask = ~active_mask
        if inactive_mask.sum() > 10:
            inactive_mean = feature_vals[inactive_mask].mean()
            inactive_median = np.median(feature_vals[inactive_mask])
        else:
            inactive_mean = inactive_median = np.nan
    else:
        active_mean = active_median = active_min = active_max = np.nan
        active_25th = active_75th = np.nan
        inactive_mean = inactive_median = np.nan
    
    # Learned threshold (approximate from data)
    # If gate is more active for higher values, threshold is around the boundary
    if not np.isnan(active_median) and not np.isnan(inactive_median):
        if active_median > inactive_median:
            direction = ">"
            # Threshold is somewhere between inactive max and active min
            estimated_threshold = (active_25th + inactive_median) / 2 if inactive_mask.sum() > 0 else active_25th
        else:
            direction = "<"
            estimated_threshold = (active_75th + inactive_median) / 2 if inactive_mask.sum() > 0 else active_75th
    else:
        direction = "?"
        estimated_threshold = np.nan
    
    gate_info.append({
        'feature': feature,
        'activation_rate': activation_rate,
        'active_mean': active_mean,
        'active_median': active_median,
        'active_min': active_min,
        'active_max': active_max,
        'active_25th': active_25th,
        'active_75th': active_75th,
        'inactive_mean': inactive_mean,
        'direction': direction,
        'estimated_threshold': estimated_threshold
    })

# Sort by activation rate
gate_info_sorted = sorted(gate_info, key=lambda x: x['activation_rate'], reverse=True)

print(f"{'Feature':<20} {'Activation':<12} {'Direction':<10} {'Estimated Threshold':<20} {'When Active: Median':<20}")
print(f"{'='*100}")

for info in gate_info_sorted:
    if info['activation_rate'] > 1:  # Only show gates that actually fire
        print(f"{info['feature']:<20} {info['activation_rate']:>10.1f}%  "
              f"{info['direction']:>9}  {info['estimated_threshold']:>19.6f}  "
              f"{info['active_median']:>19.6f}")

# ===== STEP 3: EXTRACT SIMPLE RULES =====
print(f"\n{'='*80}")
print("ACTIONABLE TRADING RULES FROM GATE BEHAVIOR")
print(f"{'='*80}\n")

print("Based on gate activation patterns in high-probability trades:\n")

# Get top gates (>50% activation)
important_gates = [g for g in gate_info_sorted if g['activation_rate'] > 50]

if important_gates:
    print("REQUIRED CONDITIONS (present in >50% of trades):")
    for i, gate in enumerate(important_gates, 1):
        if not np.isnan(gate['estimated_threshold']):
            print(f"  {i}. {gate['feature']} {gate['direction']} {abs(gate['estimated_threshold']):.6f}")
        else:
            print(f"  {i}. {gate['feature']} (active in {gate['activation_rate']:.1f}% of trades)")
        print(f"     Typical value when active: {gate['active_median']:.6f}")
        print(f"     Range: [{gate['active_min']:.6f}, {gate['active_max']:.6f}]")

# Gates that fire sometimes (10-50%)
moderate_gates = [g for g in gate_info_sorted if 10 < g['activation_rate'] <= 50]

if moderate_gates:
    print(f"\nOPTIONAL CONDITIONS (additional confirmation):")
    for i, gate in enumerate(moderate_gates, 1):
        if not np.isnan(gate['estimated_threshold']):
            print(f"  {i}. {gate['feature']} {gate['direction']} {abs(gate['estimated_threshold']):.6f}")
        else:
            print(f"  {i}. {gate['feature']} (active in {gate['activation_rate']:.1f}% of trades)")
        print(f"     Typical value when active: {gate['active_median']:.6f}")

# ===== STEP 4: GATE COMBINATION ANALYSIS =====
print(f"\n{'='*80}")
print("GATE COMBINATION PATTERNS")
print(f"{'='*80}\n")

# Count number of active gates per trade
n_active_gates = (hc_gates > 0.5).sum(axis=1)

print(f"Distribution of active gates in high-confidence trades:")
print(f"\n{'# Gates Active':<15} {'Count':<10} {'% of Trades':<15} {'Avg Return':<15} {'Win Rate':<10}")
print(f"{'='*70}")

for n_gates in range(1, hc_gates.shape[1] + 1):
    mask = n_active_gates == n_gates
    if mask.sum() > 0:
        count = mask.sum()
        pct = count / len(hc_gates) * 100
        avg_return = hc_returns[mask].mean() * 100
        win_rate = (hc_returns[mask] > 0).mean() * 100
        
        print(f"{n_gates:<15} {count:<10} {pct:<14.1f}% {avg_return:<14.4f}% {win_rate:<9.1f}%")

# ===== STEP 5: MOST COMMON GATE COMBINATIONS =====
print(f"\n{'='*80}")
print("TOP GATE COMBINATIONS")
print(f"{'='*80}\n")

# Convert gate activations to binary strings for counting
gate_patterns = []
for i in range(len(hc_gates)):
    active_features = [feature_cols[j] for j in range(len(feature_cols)) if hc_gates[i, j] > 0.5]
    if active_features:  # Only if at least one gate active
        pattern = tuple(sorted(active_features))
        gate_patterns.append({
            'pattern': pattern,
            'return': hc_returns[i]
        })

# Count patterns
pattern_df = pd.DataFrame(gate_patterns)
pattern_counts = pattern_df.groupby('pattern').agg({
    'return': ['count', 'mean', lambda x: (x > 0).mean()]
}).reset_index()
pattern_counts.columns = ['pattern', 'count', 'avg_return', 'win_rate']
pattern_counts = pattern_counts.sort_values('count', ascending=False)

print(f"Top 10 most common gate combinations:\n")
print(f"{'Rank':<6} {'Count':<8} {'Win Rate':<12} {'Avg Return':<12} {'Active Gates'}")
print(f"{'='*100}")

for idx, row in pattern_counts.head(10).iterrows():
    gates_str = ', '.join(row['pattern'])
    print(f"{idx+1:<6} {row['count']:<8.0f} {row['win_rate']*100:<11.1f}% "
          f"{row['avg_return']*100:<11.4f}% {gates_str}")

# ===== STEP 6: IMPLEMENTATION SUMMARY =====
print(f"\n{'='*80}")
print("IMPLEMENTATION SUMMARY")
print(f"{'='*80}\n")

print("Your gated neural network automatically learned these rules:\n")

print("STEP 1: Calculate features")
print("  " + ", ".join(feature_cols[:5]))
print("  " + ", ".join(feature_cols[5:]))

print("\nSTEP 2: Model applies learned gates")
if important_gates:
    print(f"  Most important: {important_gates[0]['feature']}")
    print(f"    → Active in {important_gates[0]['activation_rate']:.1f}% of trades")

print("\nSTEP 3: If model outputs probability > 0.70:")
print("  → Enter LONG position")
print(f"  → Expected: {(hc_returns > 0).mean()*100:.1f}% win rate")
print(f"  → Expected: {hc_returns.mean()*100:.4f}% avg return per trade")

print(f"\n{'='*80}")
print("\nKEY INSIGHT:")
print("You don't need to manually code thresholds - the model already learned them!")
print("The gates automatically activate when conditions are right.")
print(f"{'='*80}\n")

In [ ]:
"""
# ===== STEP 4: COMPUTE FEATURES FOR BTC =====
print("\n" + "="*80)
print("COMPUTING FEATURES")
print("="*80)

def compute_btc_features(df):
    """
    Compute only high_1d and dist_1d_high for BTC trading strategy.
    """
    df = df.copy()
    
    # ============================================================
    # DISTANCE TO 1D HIGH FEATURE
    # ============================================================
    # 1-day window (288 bars * 5 min = 24 hours)
    high_1d = df['close'].rolling(288).max()
    df['high_1d'] = high_1d
    df['dist_1d_high'] = (df['close'] - high_1d) / high_1d
    
    return df

# Calculate features for BTC
btc_symbol = symbols[0]
print(f"\nCalculating features for {btc_symbol.Value}...")
btc_df = dfs_by_symbol[btc_symbol].copy()

# Compute features
btc_df_features = compute_btc_features(btc_df)

# ===== ENSURE TIMEZONE AWARENESS =====
# Make sure the index is timezone-aware (UTC)
if btc_df_features.index.tz is None:
    btc_df_features.index = btc_df_features.index.tz_localize('UTC')
    print("Localized index to UTC")
else:
    print(f"Index already has timezone: {btc_df_features.index.tz}")

# ===== FILTER TO START FROM 2023-12-31 23:55 UTC =====
# This matches when QuantConnect's 25-hour warmup ends
start_time = pd.Timestamp('2023-12-31 23:55:00', tz='UTC')
btc_df_features = btc_df_features[btc_df_features.index >= start_time]
print(f"\nFiltered to start from: {start_time}")

# Update the dictionary
dfs_by_symbol[btc_symbol] = btc_df_features

# ===== VERIFICATION =====
print("\n" + "="*80)
print("FEATURE SUMMARY")
print("="*80)

final_df = dfs_by_symbol[btc_symbol]
print(f"Total rows: {len(final_df):,}")
print(f"Date range: {final_df.index.min()} to {final_df.index.max()}")

# List features
feature_cols = ['high_1d', 'dist_1d_high']

print("\nFeature Statistics:")
print("-" * 60)
for col in feature_cols:
    if col in final_df.columns:
        non_nan = final_df[col].notna().sum()
        nan_count = final_df[col].isna().sum()
        mean_val = final_df[col].mean()
        print(f"{col:25s} | Non-NaN: {non_nan:>8,} | NaN: {nan_count:>8,} | Mean: {mean_val:>10.6f}")

# ===== PRINT SAMPLE DATA =====
print("\n" + "="*80)
print("SAMPLE DATA (First 20 rows after 2023-12-31 23:55)")
print("="*80)

# Show first 20 rows with standard precision
sample_df = final_df[['close', 'high_1d', 'dist_1d_high']].head(20)
print(sample_df.to_string())

# ===== DETAILED DIAGNOSTICS FOR FIRST 5 BARS =====
print("\n" + "="*80)
print("DETAILED DIAGNOSTICS (First 5 bars)")
print("="*80)

for i, (timestamp, row) in enumerate(sample_df.head(5).iterrows(), 1):
    print(f"\nBar {i}:")
    print(f"  Timestamp: {timestamp}")
    print(f"  Close: ${row['close']:.2f}")
    print(f"  high_1d: ${row['high_1d']:.2f}")
    print(f"  dist_1d_high: {row['dist_1d_high']:.15f}")

# ===== DATA SOURCE INFO =====
print("\n" + "="*80)
print("DATA SOURCE INFORMATION")
print("="*80)
print(f"First timestamp in filtered data: {final_df.index[0]}")
print(f"Last timestamp in filtered data: {final_df.index[-1]}")
print(f"Total bars in filtered data: {len(final_df):,}")
print(f"Timezone: {final_df.index.tz}")
print(f"\nNote: QuantConnect uses Binance data. Ensure your pandas data is also from Binance.")
print(f"Expected bar frequency: 5 minutes")
print(f"Check if timestamps align exactly between both datasets.")

print("\n" + "="*80)
print("Columns in final dataframe:", final_df.columns.tolist())
print("="*80)
"""

In [54]:
# ===== STEP 5: FILTER FOR OBSERVATIONS AT 1-DAY HIGH =====
btc_symbol = symbols[0]
btc_df_full = dfs_by_symbol[btc_symbol].copy()

# Total observations before filtering
total_obs = len(btc_df_full)

# Filter for rows where dist_1d_high is exactly 0 (or very close due to floating point)
btc_df_at_high = btc_df_full[abs(btc_df_full['dist_1d_high']) < 1e-10].copy()

# Count observations at 1-day high
high_obs = len(btc_df_at_high)
high_pct = 100 * high_obs / total_obs if total_obs > 0 else 0

# Filter for non-NaN total_return
btc_df_filtered = btc_df_at_high[btc_df_at_high['total_return'].notna()].copy()

# Count positive vs negative returns
positive_return = (btc_df_filtered['total_return'] > 0).sum()
negative_return = (btc_df_filtered['total_return'] <= 0).sum()
total_filtered = len(btc_df_filtered)

pct_positive = 100 * positive_return / total_filtered if total_filtered > 0 else 0
pct_negative = 100 * negative_return / total_filtered if total_filtered > 0 else 0

# ===== PRINT RESULTS =====
print("\n" + "="*80)
print("1-DAY HIGH ANALYSIS")
print("="*80)
print(f"Total observations before filtering: {total_obs:,}")
print(f"Observations at 1-day high: {high_obs:,}")
print(f"Percentage of total: {high_pct:.2f}%")
print(f"\nFrom filtered observations (with valid total_return):")
print(f"  Total: {total_filtered:,}")
print(f"  Positive return: {positive_return:,} ({pct_positive:.2f}%)")
print(f"  Negative return: {negative_return:,} ({pct_negative:.2f}%)")
print("="*80)

In [ ]:
# ===== STEP 6: TRAIN DECISION TREE ON FILTERED DATA (AT 1-DAY HIGH) =====
print("\n" + "="*80)
print("TRAINING DECISION TREE ON FILTERED DATA (dist_1d_high == 0)")
print("="*80)

from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import _tree

# Use the filtered dataframe from previous step
btc_at_high = btc_df_at_high.copy()

# Define feature columns (exclude target and dist_1d_high since it's constant)
feature_cols = [
    'bb_width_2h', 'bb_width_4h', 'close_position',
    'dist_1d_low', 'dist_2h_high', 'higher_high_3',
    'hl_range', 'rvol_3', 'rvol_ratio_24_288'
]

# Create target: 1 if total_return > 0, else 0
btc_at_high['target'] = (btc_at_high['total_return'] > 0).astype(int)

# Drop rows with NaN values
btc_clean = btc_at_high.dropna()

print(f"Total samples after dropping NaN: {len(btc_clean):,}")
print(f"Positive samples (return > 0): {(btc_clean['target'] == 1).sum():,}")
print(f"Negative samples (return <= 0): {(btc_clean['target'] == 0).sum():,}")

# Check if we have enough samples
if len(btc_clean) < 100:
    print("\nWARNING: Very few samples for training. Results may not be reliable.")

# Prepare X and y
X = btc_clean[feature_cols]
y = btc_clean['target']

# Calculate class weights
n_samples = len(y)
n_positive = (y == 1).sum()
n_negative = (y == 0).sum()

class_weight_dict = {
    0: n_samples / (2 * n_negative),
    1: n_samples / (2 * n_positive)
}

print(f"\nClass weights: {class_weight_dict}")

# Train decision tree
print("\nTraining decision tree...")
tree = DecisionTreeClassifier(
    max_depth=2,
    min_samples_split=100,
    min_samples_leaf=50,
    min_impurity_decrease=0.0001,
    max_features='sqrt',
    class_weight=class_weight_dict,
    random_state=42
)

tree.fit(X, y)

print(f"Tree trained successfully!")
print(f"Number of leaves: {tree.get_n_leaves()}")
print(f"Tree depth: {tree.get_depth()}")

# ===== EXTRACT AND PRINT RULES =====
print("\n" + "="*80)
print("DECISION TREE RULES (FOR OBSERVATIONS AT 1-DAY HIGH)")
print("="*80)

def get_rules(tree, feature_names, X_data, y_data, class_names=['Negative', 'Positive']):
    """Extract rules from decision tree with actual sample counts"""
    tree_ = tree.tree_
    feature_name = [
        feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined!"
        for i in tree_.feature
    ]
    
    # Get leaf predictions for all samples
    leaf_ids = tree.apply(X_data)
    
    rules = []
    
    def recurse(node, depth, conditions, node_samples_mask):
        indent = "  " * depth
        
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            # Internal node
            name = feature_name[node]
            threshold = tree_.threshold[node]
            feature_idx = tree_.feature[node]
            
            # Split samples
            left_mask = node_samples_mask & (X_data.iloc[:, feature_idx] <= threshold)
            right_mask = node_samples_mask & (X_data.iloc[:, feature_idx] > threshold)
            
            # Left branch (<=)
            left_conditions = conditions + [f"{name} <= {threshold:.6f}"]
            recurse(tree_.children_left[node], depth + 1, left_conditions, left_mask)
            
            # Right branch (>)
            right_conditions = conditions + [f"{name} > {threshold:.6f}"]
            recurse(tree_.children_right[node], depth + 1, right_conditions, right_mask)
        else:
            # Leaf node - use actual sample counts
            node_y = y_data[node_samples_mask]
            samples = len(node_y)
            
            if samples > 0:
                n_negative = (node_y == 0).sum()
                n_positive = (node_y == 1).sum()
                proba_positive = n_positive / samples
                
                # Determine prediction based on majority
                class_idx = 1 if n_positive > n_negative else 0
                class_label = class_names[class_idx]
                
                rule = {
                    'conditions': ' AND '.join(conditions) if conditions else 'ROOT',
                    'prediction': class_label,
                    'samples': samples,
                    'class_distribution': [n_negative, n_positive],
                    'probability': proba_positive
                }
                rules.append(rule)
    
    # Start recursion with all samples
    initial_mask = pd.Series([True] * len(X_data), index=X_data.index)
    recurse(0, 0, [], initial_mask)
    return rules

# Get all rules with actual counts
rules = get_rules(tree, feature_cols, X, y, class_names=['Return <= 0', 'Return > 0'])

# Print rules
print(f"\nTotal number of rules (leaf nodes): {len(rules)}\n")

for i, rule in enumerate(rules, 1):
    print(f"Rule {i}:")
    print(f"  Conditions: {rule['conditions']}")
    print(f"  Prediction: {rule['prediction']}")
    print(f"  Samples: {rule['samples']:,}")
    print(f"  Distribution [Negative, Positive]: {rule['class_distribution']}")
    print(f"  P(Return > 0): {rule['probability']:.4f}")
    print()

# ===== FEATURE IMPORTANCE =====
print("\n" + "="*80)
print("FEATURE IMPORTANCE (AT 1-DAY HIGH)")
print("="*80)

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': tree.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance.to_string(index=False))

# ===== TRAINING ACCURACY =====
print("\n" + "="*80)
print("TRAINING METRICS (AT 1-DAY HIGH)")
print("="*80)

y_pred = tree.predict(X)
accuracy = (y_pred == y).mean()

print(f"Training Accuracy: {accuracy:.4f}")
print(f"\nPrediction Distribution:")
print(f"  Predicted Positive: {(y_pred == 1).sum():,}")
print(f"  Predicted Negative: {(y_pred == 0).sum():,}")

# ===== SAVE CLEANED DATA FOR FURTHER ANALYSIS =====
btc_clean_at_high = btc_clean.copy()
print(f"\nCleaned data saved with {len(btc_clean_at_high):,} observations")